## 비지도 학습
정답(`y`) 없이 입력 데이터(피처, `X`)의 구조와 패턴을 학습하는 방법

| 구분 | 지도학습 | 비지도학습 |
|---|---|---|
|**데이터**| 입력 + 정답 (`fit(X,y)`) | 입력만 (`fit(X)`) |
|**목표**| 정답 예측/분류 | 숨은 패턴/구조 발견 |
|**평가**| 정답과 비교 (정확도, f1 등) | 정량 지표(실루엣 등) + 도메인 해석 |

### 대표적인 유형
- **군집화(Clustering)** : 비슷한 특성의 데이터끼리 그룹화
- **차원 축소(Dimensionality Reduction)** : 많은 피처를 적은 수의 핵심 축으로 압축 -> 다중공선성 완화 및 시각화
- **이상 탐지(Anomaly Detection)** : 정상 범위에서 벗어난 패턴 감지 . 이상 거래 및 급변 탐지

> 거리, 분산 기반 알고리즘이 많으므로 **스케일링이 필수**이며,
> 군집의 의미는 최종적으로 **도메인 지식**으로 검증해야 함!

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from _style import setup
from _ml import load_merged, SEED

setup()
pd.set_option("display.width", 130)

print("준비 완료")

준비 완료


In [2]:
# 120개 종목을 수익률 패턴으로 군집화하고 실제 섹터와 비교

df = load_merged()

# 일간 수익률 피처 추가
df["ret"] = df.groupby("code")["close"].pct_change()

# 데이터를 (종목 * 날짜) 행렬로 만들기
#   행 = 날짜, 열 = 종목, 값 = 수익률
pivot = df.pivot_table(index="date", columns="code", values="ret")

# 종목별 섹터
sector = df[["code", "sectorName"]].drop_duplicates().set_index("code")["sectorName"]
true_labels = sector[pivot.columns].values


print(f"수익률 행렬 : {pivot.shape} (날짜 x 종목)")
print(f"군집화 입력 : {pivot.T.shape} (종목 x 날짜)")
print()
print("종목을 묶을 것이므로 '종목이 행'이 되도록 전치(.T)한다.")
print(f"각 종목이 {pivot.shape[0]}개의 값으로 표현된다. (날짜별 수익률)")
print()
print(f"섹터 {len(set(true_labels))}종류, 종목({len(true_labels)})개")
print(pd.DataFrame(true_labels).value_counts())

수익률 행렬 : (749, 120) (날짜 x 종목)
군집화 입력 : (120, 749) (종목 x 날짜)

종목을 묶을 것이므로 '종목이 행'이 되도록 전치(.T)한다.
각 종목이 749개의 값으로 표현된다. (날짜별 수익률)

섹터 10종류, 종목(120)개
0    
금융       18
전기전자     16
IT서비스    14
유통       12
화학       12
바이오      12
운수장비     12
건설       10
에너지       8
식품        6
Name: count, dtype: int64


---
### K-Means 군집화
k개의 평균을 찾는 알고리즘

1. k개의 중심점을 선정 (무작위로 선정)
2. 각 데이터를 가장 가까운 중심점에 배정
3. 각 그룹의 평균으로 중심점을 이동
4. 중심점이 움직이지 않을 때까지 2, 3번 반복

In [60]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score, davies_bouldin_score

def clustering(matrix, k=10, scale=True):
    """ 군집화하여 결과를 반환 """

    M = matrix.T.values

    if scale:
        M = StandardScaler().fit_transform(M)

    # km = KMeans(n_clusters=k)
    km = KMeans(
        n_clusters=k,
        random_state=SEED,      # 재현성
        n_init=10,              # n(10)번 시도해서 가장 좋은 결과를 고름
    )

    pred = km.fit_predict(M)

    return pred, M, km      # (예측결과, 입력데이터(X), 모델)

preds, M_scaled, km = clustering(pivot, k=10, scale=True)

print("군집별 종목 수 : ")
print(pd.Series(preds).value_counts().sort_index().to_string())

군집별 종목 수 : 
0     5
1    18
2    25
3    14
4     2
5    15
6    13
7     7
8    12
9     9


**`random_state`, `n_init` 을 둘 다 지정해야, 매번 실행했을 때 동일한 결과를 확인할 수 있음(재현성)**
설정하지 않으면, 중심점을 무작위로 찍고 시작하기 때문에, 실행마다 다른 군집이 나올 수 있음.

- random_state=42  : 랜덤 시드를 동일하게 주어, 항상 같은 곳을 찍게함.
- n_init=10 : 여러 번 시도해서 가장 좋은 결과를 고름.

---